# ETL Gold — Vendas Amazon

Camada **gold** do ETL de vendas: parte das tabelas **silver** (`workspace.silver.*`, construídas no notebook "ETL Silver Vendas") e produz tabelas prontas para análise em `workspace.gold`:

* `product_stock` — dimensão de produto/estoque, **1 linha por SKU** (deduplica os 6 sku_codes com múltiplas linhas no silver)
* `sales_consolidated` — fato de vendas unificando as duas fontes (Amazon India mar–jun/2022 + vendas internacionais jun/2021–mai/2022), com categoria normalizada (maiúsculas)
* `sales_monthly_summary` — agregado mensal por fonte: receita bruta/líquida, unidades, pedidos e taxa de cancelamento

⚠️ A lista de preços (`product_prices`) **não entra no gold**: testamos as chaves `sku` e `style` e não há nenhum match com as vendas (catálogos diferentes — 0 sobreposições).

In [0]:
%sql
-- Cria o schema gold (ignora se já existir)
CREATE SCHEMA IF NOT EXISTS workspace.gold

## 1. Dimensão produto/estoque

Colapsa `workspace.silver.stock` para **1 linha por SKU**:
* No silver há 6 `sku_codes` com mais de uma linha — o `GROUP BY` deduplica somando o estoque
* `MAX(category)` / `MAX(size)` / `MAX(color)`: nos casos duplicados as descrições são idênticas
* 1 SKU tem `sku_code` nulo no silver e fica em um grupo próprio (decidir se exclui na análise)

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.product_stock AS
SELECT
  sku_code                 AS sku,
  UPPER(MAX(category_norm)) AS category,
  MAX(size)                AS size,
  MAX(color)               AS color,
  SUM(stock)               AS total_stock
FROM workspace.silver.stock
GROUP BY sku_code

num_affected_rows,num_inserted_rows


## 2. Fato de vendas consolidado

Une as duas fontes com um esquema comum (`source` = `AMAZON` | `INTERNACIONAL`):

* **Categoria**: a Amazon tem categoria própria nas linhas (mantida); as vendas internacionais **herdam a categoria do estoque** via join por SKU — 1.807 das 18.635 linhas internacionais não encontram SKU no estoque e ficam com categoria nula
* As fontes **discordam** de categoria para o mesmo SKU (ex.: `JNE3797-KR-L` é `Western Dress` na Amazon e `KURTA` no estoque) — por isso o estoque só complementa a internacional, nunca sobrescreve a Amazon
* Categorias em MAIÚSCULAS para comparar entre fontes
* `unit_price`: Amazon = amount/qty (qty>0); internacional = `rate` (já é preço unitário)
* Internacional não tem status/pedido → `status` nulo, `is_cancelled` = false, `net_amount` = `gross_amt`
* Valores de referência: receita bruta Amazon ₹ 78,6M / líquida ₹ 71,7M; internacional bruta ₹ 15,8M

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.sales_consolidated AS
SELECT
  'AMAZON'                                 AS source,
  order_id,
  order_date,
  status,
  is_cancelled,
  sku,
  style,
  UPPER(TRIM(category))                    AS category,
  size,
  CAST(qty AS DOUBLE)                      AS qty,
  CASE WHEN qty > 0 THEN amount / qty END   AS unit_price,
  amount                                   AS gross_amount,
  net_amount,
  currency,
  ship_state,
  CAST(NULL AS STRING)                     AS customer,
  b2b
FROM workspace.silver.sales_amazon

UNION ALL

SELECT
  'INTERNACIONAL',
  CAST(NULL AS STRING),
  i.order_date,
  CAST(NULL AS STRING),
  false,
  i.sku,
  i.style,
  s.category,
  i.size,
  i.pcs,
  i.rate,
  i.gross_amt,
  i.gross_amt,
  'INR',
  CAST(NULL AS STRING),
  i.customer,
  CAST(NULL AS BOOLEAN)
FROM workspace.silver.sales_international i
LEFT JOIN workspace.gold.product_stock s ON i.sku = s.sku

num_affected_rows,num_inserted_rows


## 3. Resumo mensal por fonte

Agregado pronto para dashboards: receita bruta/líquida, unidades, linhas e pedidos por mês e fonte, além da **taxa de cancelamento** (valor cancelado / valor bruto).

Nota: `orders` (pedidos distintos) só existe para a Amazon — a internacional não tem número de pedido no arquivo.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.sales_monthly_summary AS
SELECT
  TRUNC(order_date, 'MM')                    AS month,
  source,
  COUNT(*)                                   AS order_lines,
  COUNT(DISTINCT order_id)                    AS orders,
  SUM(qty)                                   AS units,
  ROUND(SUM(gross_amount), 2)                AS gross_revenue,
  ROUND(SUM(net_amount), 2)                  AS net_revenue,
  ROUND(100 * SUM(CASE WHEN is_cancelled THEN gross_amount ELSE 0 END)
        / NULLIF(SUM(gross_amount), 0), 2)    AS cancel_rate_pct
FROM workspace.gold.sales_consolidated
GROUP BY 1, 2

num_affected_rows,num_inserted_rows


## Validação

Valores esperados (verificados em 23/11/2026):

| Checagem | Esperado |
| --- | --- |
| Linhas de `sales_consolidated` | 147.610 (= 128.975 + 18.635) |
| Linhas de `product_stock` | 9.171 |
| Linhas de `sales_monthly_summary` | 16 (8 meses internacional + 4 Amazon + 2 meses com ambas) |
| Receita bruta AMAZON | ₹ 78.592.678,30 |
| Receita líquida AMAZON | ₹ 71.673.394,00 |
| Receita bruta INTERNACIONAL | ₹ 15.768.162,19 |
| Linhas internacionais sem categoria | 1.807 |

In [0]:
%sql
-- Contagem por tabela
SELECT 'product_stock' AS tabela, COUNT(*) AS linhas FROM workspace.gold.product_stock
UNION ALL SELECT 'sales_consolidated', COUNT(*) FROM workspace.gold.sales_consolidated
UNION ALL SELECT 'sales_monthly_summary', COUNT(*) FROM workspace.gold.sales_monthly_summary

tabela,linhas
product_stock,9171
sales_consolidated,147610
sales_monthly_summary,16


In [0]:
%sql
-- Valores por fonte (conferir receitas e datas)
SELECT
  source,
  COUNT(*)                     AS linhas,
  ROUND(SUM(gross_amount), 2)  AS receita_bruta,
  ROUND(SUM(net_amount), 2)    AS receita_liquida,
  SUM(CASE WHEN category IS NULL THEN 1 ELSE 0 END) AS linhas_sem_categoria,
  MIN(order_date)              AS data_min,
  MAX(order_date)              AS data_max
FROM workspace.gold.sales_consolidated
GROUP BY source

source,linhas,receita_bruta,receita_liquida,linhas_sem_categoria,data_min,data_max
AMAZON,128975,7.85926783E7,7.1673394E7,0,2022-03-31,2022-06-29
INTERNACIONAL,18635,1.576816219E7,1.576816219E7,1807,2021-06-05,2022-05-11


## Análises possíveis a partir do gold

* Receita por mês e fonte (`sales_monthly_summary`)
* Top categorias / estados / clientes (`sales_consolidated`)
* Cobertura de estoque dos SKUs mais vendidos (`product_stock`)
* Evolução da taxa de cancelamento (Amazon)

## Análise — Receita por categoria

Consulta analítica sobre `sales_consolidated`:

* Linhas internacionais sem categoria aparecem como `SEM CATEGORIA` (1.807 linhas, ~2% da receita) — mantidas separadas, não misturadas com categorias reais
* Atenção à nomenclatura: a Amazon usa `SET` e o estoque (fonte da categoria internacional) usa `KURTA SET` — são taxonomias diferentes das duas fontes, não somáveis diretamente

In [0]:
%sql
-- Receita por categoria, todas as fontes
SELECT
  COALESCE(category, 'SEM CATEGORIA')                                AS categoria,
  COUNT(*)                                                         AS linhas,
  SUM(qty)                                                         AS unidades,
  ROUND(SUM(gross_amount), 2)                                      AS receita_bruta,
  ROUND(SUM(net_amount), 2)                                        AS receita_liquida,
  ROUND(100 * SUM(gross_amount) / SUM(SUM(gross_amount)) OVER (), 2) AS pct_receita
FROM workspace.gold.sales_consolidated
GROUP BY 1
ORDER BY receita_bruta DESC

categoria,linhas,unidades,receita_bruta,receita_liquida,pct_receita
SET,51478,46540.0,4.045721203E7,3.6984761E7,42.88
KURTA,57055,54112.0,2.58891767E7,2.40155E7,27.44
WESTERN DRESS,15500,13943.0,1.121607269E7,1.020959E7,11.89
TOP,12312,12224.0,6410028.3,5966302.0,6.79
KURTA SET,4249,5169.0,4987614.0,4987614.0,5.29
SEM CATEGORIA,1807,2675.0,1883321.19,1883321.19,2.0
DRESS,944,1167.0,917458.0,917458.0,0.97
ETHNIC DRESS,1159,1053.0,791217.66,732744.0,0.84
BLOUSE,1088,1028.0,517339.18,477320.0,0.55
TUNIC,638,748.0,360490.0,360490.0,0.38


In [0]:
%sql
-- Receita por categoria e fonte (AMAZON x INTERNACIONAL)
SELECT
  COALESCE(category, 'SEM CATEGORIA') AS categoria,
  source,
  COUNT(*)                            AS linhas,
  ROUND(SUM(gross_amount), 2)         AS receita_bruta,
  ROUND(SUM(net_amount), 2)           AS receita_liquida
FROM workspace.gold.sales_consolidated
GROUP BY 1, 2
ORDER BY receita_bruta DESC

categoria,source,linhas,receita_bruta,receita_liquida
SET,AMAZON,50284,3.920412403E7,3.5731673E7
KURTA,AMAZON,49877,2.12995467E7,1.942587E7
WESTERN DRESS,AMAZON,15500,1.121607269E7,1.020959E7
TOP,AMAZON,10622,5347792.3,4904066.0
KURTA SET,INTERNACIONAL,4249,4987614.0,4987614.0
KURTA,INTERNACIONAL,7178,4589630.0,4589630.0
SEM CATEGORIA,INTERNACIONAL,1807,1883321.19,1883321.19
SET,INTERNACIONAL,1194,1253088.0,1253088.0
TOP,INTERNACIONAL,1690,1062236.0,1062236.0
DRESS,INTERNACIONAL,944,917458.0,917458.0



## Resumo do projeto — fórmula X-Y-Z

> **X** (contexto) · **Y** (o que fizemos) · **Z** (resultado)

### X — De onde partimos

O ponto de partida eram **planilhas de vendas de um e-commerce de moda (Amazon India)**: vendas Amazon (mar–jun/2022, com status, categoria, estado e valor líquido), vendas internacionais (jun/2021–mai/2022, por cliente e sem número de pedido), uma planilha de **estoque por SKU** e uma **lista de preços**. Os dados estavam crus: categorias com grafias diferentes entre fontes, 6 SKUs duplicados no estoque, linhas internacionais sem categoria e taxonomias que não batiam entre si (ex.: `SET` na Amazon x `KURTA SET` no estoque).

### Y — O que foi feito

1. **Silver**: limpeza e normalização dos dados brutos — categorias normalizadas, estoque deduplicado por `sku_code`, tipagem de datas e valores, marcando cancelamentos (`is_cancelled`).
2. **Gold**: três tabelas prontas para análise em `workspace.gold`:
   * `product_stock` — 1 linha por SKU (9.171 SKUs), com categoria, tamanho, cor e estoque total;
   * `sales_consolidated` — fato unificada das duas fontes (147.610 linhas), com esquema comum (`source` = AMAZON | INTERNACIONAL), categoria herdada do estoque apenas para a fonte internacional e preços unitários padronizados;
   * `sales_monthly_summary` — 16 meses agregados por fonte: receita bruta/líquida, unidades, pedidos e taxa de cancelamento.
3. **Validação**: contagens e receitas conferidas; a lista de preços ficou **de fora** após testes de join (`sku` e `style`) — 0 sobreposições com as vendas.

### Z — Resultado

De planilhas dispersas e inconsistentes para um **modelo analítico confiável e pronto para dashboards**: receita bruta Amazon de **₹ 78,6 M** (líquida **₹ 71,7 M**) e internacional de **₹ 15,8 M** consolidadas na mesma tabela, com métricas mensais e taxa de cancelamento por fonte. Os 1.807 registros internacionais sem categoria (~2% da receita) foram mantidos e sinalizados como `SEM CATEGORIA` em vez de descartados, preservando a integridade dos totais.

